1.Data Understanding

In [3]:
import pandas as pd

#load dataset
df = pd.read_csv("/content/sample_data/IMDB Dataset.csv")

#first 5 rows
print(df.head())

#number Of samples
print("\nShape:", df.shape)

#class distribution
print("\nclass Distribution:")
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Shape: (50000, 2)

class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


2.NLP Preprocessing

In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem

nltk.download('stopwords')

# Initialize
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # 1. Lowercasing
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # 3. Remove special characters & punctuation
    text = re.sub(r'[^a-z\s]', '', text)

    # 4. Tokenization
    words = text.split()

    # 5. Remove stopwords
    words = [word for word in words if word not in stop_words]

    # 6. Stemming
    words = [stemmer.stem(word) for word in words]

    return " ".join(words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
df['clean_text'] = df['review'].apply(preprocess_text)
print(df[['review', 'clean_text']].head())

                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                          clean_text  
0  one review mention watch oz episod youll hook ...  
1  wonder littl product br br film techniqu unass...  
2  thought wonder way spend time hot summer weeke...  
3  basic there famili littl boy jake think there ...  
4  petter mattei love time money visual stun film...  


3.Feature Engineering

Convert text using Bag of Words (BoW)

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize BoW
bow = CountVectorizer(max_features=5000)

# Convert text to vectors
X_bow = bow.fit_transform(df['clean_text']).toarray()

print("BoW Shape:", X_bow.shape)

BoW Shape: (50000, 5000)


Convert text using TF-IDF.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF
tfidf = TfidfVectorizer(max_features=5000)

# Convert text
X_tfidf = tfidf.fit_transform(df['clean_text']).toarray()

print("TF-IDF Shape:", X_tfidf.shape)

TF-IDF Shape: (50000, 5000)


4.Model Building

Train-Test split

In [15]:
from sklearn.model_selection import train_test_split

X = X_tfidf   # use TF-IDF features
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train Size:", X_train.shape)
print("Test Size:", X_test.shape)

Train Size: (40000, 5000)
Test Size: (10000, 5000)


Logistic Regression

In [21]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)


Logistic Regression
Accuracy : 0.8841
Precision: 0.8844
Recall   : 0.8841
F1 Score : 0.8841
Logistic Regression Training Done


Naive Bayes

In [17]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

print("Naive Bayes Training Done")

Naive Bayes Training Done


Decision Tree

In [18]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

print("Decision Tree Training Done")

Decision Tree Training Done


5.Model Evaluation

Evaluate using Accuracy, Precision, Recall, and F1 Score

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    print(f"\n{name}")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(prec, 4))
    print("Recall   :", round(rec, 4))
    print("F1 Score :", round(f1, 4))

    return acc, prec, rec, f1

In [20]:
results = {}

results['Logistic Regression'] = evaluate_model("Logistic Regression", lr, X_test, y_test)
results['Naive Bayes'] = evaluate_model("Naive Bayes", nb, X_test, y_test)
results['Decision Tree'] = evaluate_model("Decision Tree", dt, X_test, y_test)


Logistic Regression
Accuracy : 0.8841
Precision: 0.8844
Recall   : 0.8841
F1 Score : 0.8841

Naive Bayes
Accuracy : 0.8458
Precision: 0.8458
Recall   : 0.8458
F1 Score : 0.8458

Decision Tree
Accuracy : 0.717
Precision: 0.717
Recall   : 0.717
F1 Score : 0.717


Compare performance

In [22]:
import pandas as pd

comparison_df = pd.DataFrame(results, index=["Accuracy", "Precision", "Recall", "F1 Score"]).T

print("\nModel Comparison:\n")
print(comparison_df)


Model Comparison:

                     Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.8841   0.884423  0.8841  0.884060
Naive Bayes            0.8458   0.845830  0.8458  0.845788
Decision Tree          0.7170   0.717019  0.7170  0.717004


6.Comparison & Insights

Best Preprocessing Steps:

Lowercasing is used for the removing duplicate words.

Removing punctuation and special characters cleaned the text.

Stopword removal for reduced unnecessary words.

Stemming for improved consistency of words.

These steps made the data clean and suitable for ML models.

Best Vectorization Technique:

TF-IDF performed better than Bag of Words(BoW).

TF-IDF gives more importance to meaningful words and less to common words.

Therefore, TF-IDF improved model accuracy.

Best Model:

Logistic Regression performed the best.

It gave highest accuracy and balanced Precision, Recall, and F1 Score.

It works well for large text datasets.

Model Comparison:
Logistic Regression: High accuracy, reliable.

Naive Bayes: Fast but slightly less accurate.

Decision Tree: Lower performance due to overfitting.

Trade-offs:

Method:BoW

Advantage:Simple,fast.

Disadvantage:No importance to words.

Method:TF-IDF

Advantage:Better accuracy.

Disadvantage:Slightly slower.

Method:Logistic Regression

Advantage:High accuracy.

Disadvantage:Needs tuning.

Method:Naive Bayes.

Advantage:Very fast.

Disadvantage:Assumes independence.

Method:Decision Tree

Advantage:Easy to understand.

Disadvantage:Overfitting problem.